In [6]:
!pip install -q \
    chromadb \
    google-genai \
    groq

print("Libraries installed successfully!")

Libraries installed successfully!


In [5]:
import os

print("chunks.pkl exists:", os.path.exists("chunks.pkl"))
print("chunk_embeddings.pkl exists:", os.path.exists("chunk_embeddings.pkl"))

chunks.pkl exists: False
chunk_embeddings.pkl exists: False


In [7]:
import pickle

try:
    with open("chunks.pkl", "rb") as f:
        chunks = pickle.load(f)

    with open("chunk_embeddings.pkl", "rb") as f:
        chunk_embeddings = pickle.load(f)

    print("✅ Files loaded successfully!")

except FileNotFoundError as e:
    print(f"❌ File not found: {e}")

except Exception as e:
    print(f"❌ Loading Error: {e}")

✅ Files loaded successfully!


In [8]:
import chromadb

try:
    chroma_client = chromadb.PersistentClient(
        path="./networking_chromadb"
    )

    collection = chroma_client.get_or_create_collection(
        name="networking_docs"
    )

    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=chunk_embeddings
    )

    print("✅ Collection recreated successfully!")
    print("📊 Records:", collection.count())

except Exception as e:
    print("❌ ChromaDB Error:")
    print(e)

✅ Collection recreated successfully!
📊 Records: 247


In [9]:
from google import genai
from google.colab import userdata

try:
    client = genai.Client(
        api_key=userdata.get("GEMINI_KEY")
    )

    print("✅ Gemini initialized successfully!")

except Exception as e:
    print("❌ Gemini Error:")
    print(e)

✅ Gemini initialized successfully!


In [10]:
try:

    query = "What is Machine Learning?"

    response = client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=query
    )

    query_embedding = response.embeddings[0].values

    print("Query embedding generated!")

except Exception as e:

    print("Embedding Error:")
    print(e)

Query embedding generated!


In [11]:
try:

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    print("Retrieved top 5 chunks!")

except Exception as e:

    print("Retrieval Error:")
    print(e)


Retrieved top 5 chunks!


In [12]:
try:

    retrieved_chunks = results["documents"][0]

    for i, chunk in enumerate(retrieved_chunks):

        print("=" * 60)
        print(f"Chunk {i + 1}")
        print(chunk[:500])
        print()

except Exception as e:

    print("Display Error:")
    print(e)

Chunk 1
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
Statistics and mathematical optimisation methods compose the foundations o

Chunk 2
Models
A machine learning model is a type of mathematical model that, once "trained" on a given dataset, can be used to make predictions or classifications on new data. During training, a learning algorithm iteratively adjusts the model's internal parameters to minimise errors in its predictions. By extension, the term "model" can refer to several levels of specificity, from a general class of models and their associated learning algorithms to a fully trained model with all its

In [14]:
from groq import Groq
from google.colab import userdata

try:
    groq_client = Groq(
        api_key=userdata.get("GROQ_KEY")
    )

    print("✅ Groq initialized successfully!")

except Exception as e:
    print("❌ Groq Error:")
    print(e)

✅ Groq initialized successfully!


In [15]:
try:

    context = "\n\n".join(retrieved_chunks)

    print("Context created successfully!")

except Exception as e:

    print("Context Error:")
    print(e)

Context created successfully!


In [16]:

try:

    prompt =f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have enough information."
Context:
{context}

Question:
{query}
"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    print(answer)

except Exception as e:

    print("Generation Error:")
    print(e)

Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed.


In [20]:

def ask_question(question):

    try:

        embedding_response = client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=question
        )

        query_embedding = embedding_response.embeddings[0].values

        search_results = collection.query(
            query_embeddings=[query_embedding],
            n_results=5
        )

        context = "\n\n".join(
            search_results["documents"][0]
        )

        prompt = f"""
You are a helpful assistant.

Answer only using the provided context.

Context:
{context}

Question:
{question}
"""

        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return response.choices[0].message.content

    except Exception as e:

        return f"Error: {e}"

In [21]:
print(
    ask_question(
        "What is a neural network?"
    )
)

A neural network is a group of interconnected units called neurons that send signals to one another. Neurons can be either biological cells or mathematical models.
